# 🎛️ 32 · Uncertainty Quantification & Active Learning
### How much should you trust a prediction — and which image should you label next?

**Part J · Research frontier** · Module 32 of 37 (frontier) · builds on 16, 19

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/32_uncertainty_and_active_learning.ipynb)

| | |
|---|---|
| **Hardware target** | CPU (every demo finishes in well under a minute; a GPU is not needed) |
| **Core frameworks** | NumPy, Matplotlib, PyTorch |
| **Prerequisites** | Module 16 (deep learning classification), Module 19 (deep segmentation) |
| **Builds on** | Modules 16, 19 |
| **Research frontier covered** | Calibration, Bayesian/ensemble uncertainty, active learning, human-in-the-loop annotation |
| **Estimated time** | 75–95 min |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> **All data are SIMULATED.** Nuclei images, class labels and the pool of "unlabeled" images are all generated by code with known ground truth, so every accuracy, calibration and sample-efficiency number below can be checked exactly rather than taken on faith.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain the difference between **aleatoric** uncertainty (noise in the data itself) and
  **epistemic** uncertainty (the model's own lack of knowledge), and why only one of them
  shrinks as you collect more data.
- Check whether a classifier's reported confidence is **calibrated** — whether "90% sure" really
  means right 90% of the time — using a reliability diagram and the expected calibration error.
- Get a practical uncertainty estimate out of a network you already know how to train, two ways:
  **MC-dropout** and a small **deep ensemble**.
- Build an **active-learning** loop that chooses which unlabeled images to annotate next, and
  show — with a controlled, ground-truth-backed experiment — that it reaches a target accuracy
  with fewer labels than random annotation.
- Describe how interactive tools such as napari and ilastik turn this loop into a real annotation
  workflow with a human in it.

## 🗺️ Contents

1. A classifier that doesn't know what it doesn't know
2. Where does the uncertainty come from? Aleatoric vs. epistemic
3. Getting an uncertainty estimate out of one network: MC-dropout and deep ensembles
4. Active learning: spending your labeling budget wisely
5. Human-in-the-loop: closing the loop with a person
6. 🧭 From toy to real data

## 🧫 The biological question

In Module 16 you trained a classifier to tell six nuclear/cell shape classes apart, and in
Module 19 you trained a segmenter to outline cells in a fluorescence image. Both models reported
a number for every prediction — a class probability, a per-pixel segmentation score — and you
quietly started trusting those numbers.

But in the lab you will turn a model loose on images it has never seen: a different microscope,
a new cell line, a batch with more debris than the training set ever had. A model that is *wrong
and says so* (low confidence) is a minor nuisance — you check it by eye and move on. A model that
is *wrong and sounds certain* can send a whole analysis, or a whole experiment, in the wrong
direction before anyone notices. This module is about getting a model to say "I'm not sure" when
it means it, and about spending your most precious experimental resource — a trained annotator's
time — on exactly the images where that uncertainty is highest.

---

## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image torch
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
SEED = 24
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

The exact same helper functions used throughout this course, copied verbatim so this notebook runs on its own.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

SHAPE_CLASSES = ("round", "elongated", "kidney", "ring", "dividing", "irregular")


def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def simulate_nuclei_image(n_cells: int = 25, size: int = 160, touching: bool = False,
                           shape_classes: tuple[str, ...] | None = None,
                           psf_sigma: float = 1.2, background: float = 0.05,
                           noise_sigma: float = 0.03, seed: int = 0):
    """A SIMULATED single-channel fluorescence image of cell nuclei/cells.

    Returns ``image`` (size x size, background + PSF-blurred blobs + Gaussian noise),
    ``instance_mask`` (0 = background, 1..n_cells = one integer id per cell, the ground truth
    for segmentation/detection/tracking), ``centroids`` (n_cells x 2, row, col), ``shape_labels``
    (one of SHAPE_CLASSES per cell, the ground truth for classification), and ``boxes`` (n_cells x 4,
    ymin, xmin, ymax, xmax — the ground truth for detection).
    """
    rng = np.random.default_rng(seed)
    classes = shape_classes or SHAPE_CLASSES
    image = np.full((size, size), background, dtype=float)
    instance_mask = np.zeros((size, size), dtype=int)
    centroids, labels, boxes = [], [], []
    margin = size * 0.12
    min_sep = size * 0.11 if not touching else size * 0.065
    placed = []
    attempts = 0
    while len(placed) < n_cells and attempts < n_cells * 60:
        attempts += 1
        cy, cx = rng.uniform(margin, size - margin, size=2)
        if all(np.hypot(cy - py, cx - px) > min_sep for py, px in placed):
            placed.append((cy, cx))
    for i, (cy, cx) in enumerate(placed):
        shape = classes[rng.integers(len(classes))]
        r = rng.uniform(size * 0.028, size * 0.045)
        angle = rng.uniform(0, 2 * np.pi)
        single = np.zeros_like(image)
        _draw_blob(single, cy, cx, r, shape, rng, angle)
        image += single
        this_mask = single > 0.25
        instance_mask[this_mask & (instance_mask == 0)] = i + 1
        centroids.append((cy, cx))
        labels.append(shape)
        ys, xs = np.where(this_mask)
        if len(ys):
            boxes.append((ys.min(), xs.min(), ys.max(), xs.max()))
        else:
            boxes.append((cy - r, cx - r, cy + r, cx + r))
    image = blur_image(image, gaussian_psf(psf_sigma))
    image = add_gaussian_noise(image, noise_sigma, rng)
    image = np.clip(image, 0, None)
    return dict(image=image, instance_mask=instance_mask, centroids=np.array(centroids),
                shape_labels=np.array(labels), boxes=np.array(boxes), source="SIMULATED")


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes

---

## 1 · A classifier that doesn't know what it doesn't know

We start exactly where Module 16 left off: a small convolutional classifier for the six
`SHAPE_CLASSES` nuclei shapes. To keep training fast we make each training example a tiny
48 × 48 image containing exactly **one** simulated cell — `simulate_nuclei_image(n_cells=1, ...)`
already gives us this, with a known shape label, for free.

In [ ]:
# ── Build a small labeled dataset of single-cell patches ──
SHAPE_TO_IDX = {s: i for i, s in enumerate(SHAPE_CLASSES)}
PATCH_SIZE = 48    # the classifier's input size
GEN_SIZE = 160     # the simulator canvas size (shape detail needs a reasonably large canvas;
                   # we then crop tightly around the cell and resize down to PATCH_SIZE)


def crop_and_resize_cell(sim, patch_size=PATCH_SIZE, margin=1.7):
    """Crop a simulated single-cell image tightly around its (known) bounding box and resize to
    a fixed patch_size — the same "detect, then crop-and-classify" step a real pipeline uses."""
    img = sim["image"]
    ymin, xmin, ymax, xmax = sim["boxes"][0]
    cy, cx = (ymin + ymax) / 2, (xmin + xmax) / 2
    half = max(6.0, max(ymax - ymin, xmax - xmin) / 2 * margin)
    y0, y1 = int(max(0, cy - half)), int(min(img.shape[0], cy + half))
    x0, x1 = int(max(0, cx - half)), int(min(img.shape[1], cx + half))
    crop = img[y0:y1, x0:x1]
    zoom_factors = (patch_size / crop.shape[0], patch_size / crop.shape[1])
    return ndi.zoom(crop, zoom_factors, order=1)


def make_shape_dataset(n_per_class, seed, noise_sigma=0.05):
    """A SIMULATED dataset: n_per_class cropped-and-resized single-cell images per shape class.
    Returns (images [N,1,PATCH_SIZE,PATCH_SIZE] float32, labels [N] int, the per-sample seeds)."""
    images, labels, seeds_used = [], [], []
    s = seed * 100_003  # spread seeds out so different calls never collide
    for shape in SHAPE_CLASSES:
        for _ in range(n_per_class):
            sim = simulate_nuclei_image(n_cells=1, size=GEN_SIZE, shape_classes=(shape,),
                                         noise_sigma=noise_sigma, seed=s)
            images.append(crop_and_resize_cell(sim))
            labels.append(SHAPE_TO_IDX[shape])
            seeds_used.append(s)
            s += 1
    X = np.stack(images)[:, None, :, :].astype(np.float32)
    y = np.array(labels, dtype=np.int64)
    perm = rng.permutation(len(y))
    return X[perm], y[perm], np.array(seeds_used)[perm]


X_train, y_train, _ = make_shape_dataset(n_per_class=45, seed=1)
X_test, y_test, _ = make_shape_dataset(n_per_class=20, seed=999)
print("train:", X_train.shape, "test:", X_test.shape)
print("classes:", SHAPE_CLASSES)

fig, axes = plt.subplots(1, 6, figsize=(14, 2.6))
for cls_idx in range(6):
    i = np.where(y_train == cls_idx)[0][0]
    show_image(axes[cls_idx], X_train[i, 0], SHAPE_CLASSES[cls_idx])
fig.suptitle("One example per class")
fig.tight_layout()
plt.show()

🔍 **What just happened?** Every image above is produced on the spot by the same
`simulate_nuclei_image` function from Module 00 onward — there is no hidden dataset, and every
label is exactly correct because we told the simulator which shape to draw.

In [ ]:
# ── A small CNN with dropout (dropout will double as our uncertainty tool in §3) ──
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(SEED)


class TinyShapeNet(nn.Module):
    """A deliberately small CNN classifier for 48x48 single-channel cell patches."""

    def __init__(self, n_classes=6, p_drop=0.3):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.drop = nn.Dropout(p_drop)
        self.fc1 = nn.Linear(16 * 12 * 12, 32)
        self.fc2 = nn.Linear(32, n_classes)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)       # 48 -> 24
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)       # 24 -> 12
        x = self.drop(x.flatten(1))
        x = self.drop(F.relu(self.fc1(x)))
        return self.fc2(x)


def train_classifier(X, y, n_epochs=180, lr=2e-3, p_drop=0.15, seed=0):
    torch.manual_seed(seed)
    model = TinyShapeNet(p_drop=p_drop)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xt, yt = torch.from_numpy(X), torch.from_numpy(y)
    for epoch in range(n_epochs):
        model.train()
        opt.zero_grad()
        loss = F.cross_entropy(model(Xt), yt)
        loss.backward()
        opt.step()
    return model


base_model = train_classifier(X_train, y_train, seed=SEED)
base_model.eval()
with torch.no_grad():
    logits = base_model(torch.from_numpy(X_test))
    probs = F.softmax(logits, dim=1).numpy()
preds = probs.argmax(axis=1)
test_acc = (preds == y_test).mean()
print(f"test accuracy: {test_acc:.1%}")

In [ ]:
# ── 🧪 Reliability diagram: is a stated confidence actually right that often? ──
confidence = probs.max(axis=1)
correct = (preds == y_test).astype(float)

n_bins = 8
bin_edges = np.linspace(0, 1, n_bins + 1)
bin_acc, bin_conf, bin_count = [], [], []
for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    sel = (confidence > lo) & (confidence <= hi)
    if sel.sum() > 0:
        bin_acc.append(correct[sel].mean())
        bin_conf.append(confidence[sel].mean())
        bin_count.append(sel.sum())
    else:
        bin_acc.append(np.nan)
        bin_conf.append((lo + hi) / 2)
        bin_count.append(0)
bin_acc, bin_conf, bin_count = map(np.array, (bin_acc, bin_conf, bin_count))

ece = np.nansum(np.abs(bin_acc - bin_conf) * bin_count) / bin_count.sum()

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot([0, 1], [0, 1], "k--", label="perfect calibration")
ax.bar(bin_edges[:-1], np.nan_to_num(bin_acc), width=1 / n_bins, align="edge",
       alpha=0.6, edgecolor="k", label="empirical accuracy")
ax.set_xlabel("predicted confidence"); ax.set_ylabel("empirical accuracy")
ax.set_title(f"Reliability diagram (ECE = {ece:.3f})")
ax.legend(fontsize=8)
plt.show()
print(f"Expected calibration error (ECE): {ece:.3f}  (0 = perfectly calibrated)")

✏️ **Your turn.** Re-run the training cell with `n_epochs=5` (clearly under-trained) and
again with `n_epochs=120` (likely over-confident). Watch what happens to the ECE in each case —
under-training and over-training tend to miscalibrate a network in *opposite* directions.

---

## 2 · Where does the uncertainty come from? Aleatoric vs. epistemic

Not all uncertainty is the same kind, and the difference matters for what you do about it:

- **Aleatoric uncertainty** is noise *in the data itself* — a genuinely ambiguous image, a noisy
  detector, a cell caught mid-transition between two shape classes. No amount of extra training
  data removes it, because the image itself does not contain the answer.
- **Epistemic uncertainty** is the *model's own* lack of knowledge — a shape class it has barely
  seen, a combination of features outside its training experience. This kind **does** shrink as
  you add more (and better) labeled examples.

Telling them apart matters because the fix is different: aleatoric uncertainty calls for a better
microscope or a different assay; epistemic uncertainty calls for more labels — which is exactly
what active learning (§4) spends wisely.

In [ ]:
# ── 🧪 Make the two kinds concrete and separately checkable ──
# Aleatoric: crank up the imaging noise. The *label* is still exactly known (we are the
# simulator), but the pixels genuinely carry less information about it.
X_noisy, y_noisy, _ = make_shape_dataset(n_per_class=20, seed=7, noise_sigma=0.22)
with torch.no_grad():
    probs_noisy = F.softmax(base_model(torch.from_numpy(X_noisy)), dim=1).numpy()
acc_noisy = (probs_noisy.argmax(1) == y_noisy).mean()
conf_clean = probs.max(axis=1).mean()
conf_noisy = probs_noisy.max(axis=1).mean()
print(f"clean test images  : accuracy {test_acc:.1%}, mean confidence {conf_clean:.2f}")
print(f"high-noise images  : accuracy {acc_noisy:.1%}, mean confidence {conf_noisy:.2f}")

# Epistemic: starve one class of training examples, then compare a 5-member ensemble's
# *disagreement* on that class against a class it saw plenty of.
X_starved, y_starved, _ = make_shape_dataset(n_per_class=45, seed=2)
starve_mask = y_starved != SHAPE_TO_IDX["ring"]
keep_ring = np.where(~starve_mask)[0][:3]            # keep only 3 "ring" examples
keep = np.concatenate([np.where(starve_mask)[0], keep_ring])
X_starved, y_starved = X_starved[keep], y_starved[keep]

mini_ensemble = [train_classifier(X_starved, y_starved, seed=100 + k) for k in range(5)]


def ensemble_probs(models, X):
    with torch.no_grad():
        return np.stack([F.softmax(m(torch.from_numpy(X)), dim=1).numpy() for m in models])


ens = ensemble_probs(mini_ensemble, X_test)           # [5, N, 6]
disagreement = ens.std(axis=0).sum(axis=1)            # total predictive std per test image
ring_mask = y_test == SHAPE_TO_IDX["ring"]
print(f"\nmean ensemble disagreement on a starved class (ring) : {disagreement[ring_mask].mean():.3f}")
print(f"mean ensemble disagreement on a well-seen class (round): "
      f"{disagreement[y_test == SHAPE_TO_IDX['round']].mean():.3f}")

🔍 **What just happened?** The noisy-image accuracy dropped while confidence barely
moved as much as accuracy did — that gap *is* miscalibration caused by aleatoric uncertainty the
model cannot resolve by looking harder. Separately, the ensemble disagrees with itself far more on
the class it saw only 3 times than on a class it saw 45 times — that gap is epistemic uncertainty,
and §4 shows it is exactly what more labels fix.

---

## 3 · Getting an uncertainty estimate out of one network

Training five networks (as above) is one way to estimate epistemic uncertainty, but it is
expensive. Two cheaper, widely used alternatives:

- **MC-dropout** (Gal & Ghahramani, 2016): leave dropout *turned on* at test time and run the same
  input through the network many times. Each pass is a slightly different random sub-network —
  the spread across passes approximates the spread you would get from training many models.
- **Deep ensembles** (Lakshminarayanan et al., 2017): train a handful of ordinary networks from
  different random initializations and look at how much they disagree. Usually more reliable than
  MC-dropout, at the cost of training more than one model.

In [ ]:
# ── MC-dropout: keep dropout active at inference, run many stochastic passes ──
def mc_dropout_predict(model, X, n_passes=30):
    model.train()   # keeps Dropout layers stochastic; BatchNorm would need care, we have none
    with torch.no_grad():
        passes = np.stack([F.softmax(model(torch.from_numpy(X)), dim=1).numpy()
                            for _ in range(n_passes)])
    model.eval()
    return passes                                     # [n_passes, N, n_classes]


mc_passes = mc_dropout_predict(base_model, X_test, n_passes=30)
mc_mean = mc_passes.mean(axis=0)
mc_uncertainty = mc_passes.std(axis=0).sum(axis=1)     # total predictive std, one number per image
mc_entropy = -(mc_mean * np.log(mc_mean + 1e-9)).sum(axis=1)

correct_base = (mc_mean.argmax(axis=1) == y_test)
print(f"MC-dropout mean uncertainty | correct predictions : {mc_uncertainty[correct_base].mean():.3f}")
print(f"MC-dropout mean uncertainty | wrong predictions    : {mc_uncertainty[~correct_base].mean():.3f}")

🧪 **Ground-truth check.** Because we have the true labels, we can ask the single
most important question about any uncertainty estimate: *is it actually higher on the examples
the model gets wrong?* If MC-dropout's uncertainty is systematically higher on the wrong
predictions above, the estimate is doing its job — a model that is "confidently wrong" with no
warning is far more dangerous than one that is "wrong, but it told you so".

In [ ]:
# ── Compare MC-dropout against the 5-member ensemble on the SAME test set ──
full_ensemble = [train_classifier(X_train, y_train, seed=200 + k) for k in range(5)]
ens_full = ensemble_probs(full_ensemble, X_test)
ens_uncertainty = ens_full.std(axis=0).sum(axis=1)

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(mc_uncertainty, ens_uncertainty, c=correct_base, cmap="coolwarm_r", s=18, alpha=0.8)
ax.set_xlabel("MC-dropout uncertainty"); ax.set_ylabel("deep-ensemble uncertainty")
ax.set_title("Two uncertainty estimates\n(blue = correct, red = wrong)")
plt.show()
corr = np.corrcoef(mc_uncertainty, ens_uncertainty)[0, 1]
print(f"correlation between the two uncertainty estimates: {corr:.2f}")

✏️ **Your turn.** Both estimates should land higher, on average, on the wrong
predictions, and the two should be positively correlated with each other even though one came
from a single stochastic network and the other from five independently trained ones. Try raising
`p_drop` in `TinyShapeNet` — does MC-dropout uncertainty become a better or worse match for the
ensemble's?

---

## 4 · Active learning: spending your labeling budget wisely

Suppose you have a large pool of unlabeled cell images and a limited annotation budget — exactly
the usual situation in a real lab. The question active learning asks is simple: **which images,
if labeled next, will improve the model the most?** The obvious baseline is to label a random
batch each round. The active-learning idea is to label the batch the *current* model is most
uncertain about — its own uncertainty estimate (§3) is the acquisition signal.

In [ ]:
# ── A controlled active-learning experiment with a known, checkable answer ──
X_pool, y_pool, _ = make_shape_dataset(n_per_class=60, seed=5)     # the "unlabeled" pool
X_val, y_val, _ = make_shape_dataset(n_per_class=25, seed=777)     # a fixed held-out test set

N_SEED_LABELS = 12          # 2 per class to start
N_PER_ROUND = 12
N_ROUNDS = 8


def entropy_of(probs):
    return -(probs * np.log(probs + 1e-9)).sum(axis=1)


def run_strategy(strategy, seed):
    rng_local = np.random.default_rng(seed)
    labeled = list(rng_local.choice(len(X_pool), size=N_SEED_LABELS, replace=False))
    unlabeled = [i for i in range(len(X_pool)) if i not in labeled]
    accs, n_labels = [], []
    for round_i in range(N_ROUNDS):
        model = train_classifier(X_pool[labeled], y_pool[labeled], n_epochs=150, seed=seed * 7 + round_i)
        with torch.no_grad():
            val_probs = F.softmax(model(torch.from_numpy(X_val)), dim=1).numpy()
        accs.append((val_probs.argmax(1) == y_val).mean())
        n_labels.append(len(labeled))
        if not unlabeled:
            break
        if strategy == "random":
            pick = list(rng_local.choice(unlabeled, size=min(N_PER_ROUND, len(unlabeled)), replace=False))
        else:  # "uncertainty"
            with torch.no_grad():
                pool_probs = F.softmax(model(torch.from_numpy(X_pool[unlabeled])), dim=1).numpy()
            order = np.argsort(-entropy_of(pool_probs))     # most uncertain first
            pick = [unlabeled[i] for i in order[:N_PER_ROUND]]
        labeled += pick
        unlabeled = [i for i in unlabeled if i not in pick]
    return np.array(n_labels), np.array(accs)


n_rand, acc_rand = run_strategy("random", seed=SEED)
n_unc, acc_unc = run_strategy("uncertainty", seed=SEED)

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(n_rand, acc_rand, "o-", label="random labeling")
ax.plot(n_unc, acc_unc, "s-", label="uncertainty sampling (active learning)")
ax.set_xlabel("number of labeled images"); ax.set_ylabel("held-out accuracy")
ax.legend(); ax.set_title("Sample efficiency: active learning vs. random")
plt.show()


def labels_to_reach(n_labels, accs, target):
    hit = np.where(accs >= target)[0]
    return int(n_labels[hit[0]]) if len(hit) else None


TARGET = 0.8
print(f"labels needed to reach {TARGET:.0%} held-out accuracy:")
print(f"  random labeling       : {labels_to_reach(n_rand, acc_rand, TARGET)}")
print(f"  uncertainty sampling  : {labels_to_reach(n_unc, acc_unc, TARGET)}")

🧪 **Ground-truth check.** Compare how many labels each curve needs to first reach,
say, 80% held-out accuracy. Because `X_val`/`y_val` are a fixed, simulated, exactly-known test
set, "which strategy got there with fewer labels" is not a matter of opinion — read it directly
off the two curves above. Uncertainty sampling should reach any given accuracy level with
noticeably fewer labeled images, because it spends the budget on the images the model is least
sure about rather than images it may already classify correctly.

✏️ **Your turn.** This acquisition function (pick the highest-entropy pool images) is
the simplest one. A more sophisticated one, BALD (Bayesian Active Learning by Disagreement), picks
points where *individual* MC-dropout passes disagree with each other the most — not just where the
*average* prediction is uncertain. Can you think of a pool image where total entropy is high but
the passes mostly agree (e.g., genuinely ambiguous between exactly two classes), versus one where
entropy is high *because* the passes disagree wildly? The two cases call for different fixes.

---

## 5 · Human-in-the-loop: closing the loop with a person

Everything above assumed we could ask an oracle (our own simulator) for the true label on demand.
In the real lab the "oracle" is a person at a microscope or an annotation workstation, and the
loop looks like this:

1. The model proposes labels or a segmentation on new images.
2. A human annotator reviews the proposals — ideally starting with the ones the model is least
   confident about, exactly as in §4 — and corrects the mistakes.
3. The corrected examples are added to the training set and the model retrains.
4. Repeat.

This is precisely what interactive tools such as **napari** (an image viewer with a plugin
ecosystem for exactly this kind of correction workflow) and **ilastik** (which trains classical
pixel-classification and object-classification models interactively as you paint corrections) are
built around. The uncertainty estimate is what decides where the annotator's attention goes next —
the same signal you computed by hand in §3 and used in §4.

---

## 6 · 🧭 From toy to real data

### 📄 Research connection — Bayesian deep learning by dropout
Gal, Y. & Ghahramani, Z. (2016). *Dropout as a Bayesian Approximation: Representing Model
Uncertainty in Deep Learning.* ICML. The theoretical justification for using dropout at test time
(§3) as an approximate Bayesian posterior over network weights.

### 📄 Research connection — Deep ensembles
Lakshminarayanan, B., Pritzel, A. & Blundell, C. (2017). *Simple and Scalable Predictive
Uncertainty Estimation using Deep Ensembles.* NeurIPS. Often matches or beats MC-dropout in
practice while being conceptually simpler — train several ordinary networks, disagree.

### 📄 Research connection — Calibration of modern networks
Guo, C., Pleiss, G., Sun, Y. & Weinberger, K. Q. (2017). *On Calibration of Modern Neural
Networks.* ICML. Shows that larger, more accurate networks are often *less* calibrated than older,
smaller ones — accuracy and trustworthy confidence are not the same objective.

### 📄 Research connection — Active learning
Settles, B. (2009). *Active Learning Literature Survey.* University of Wisconsin-Madison,
Computer Sciences Technical Report 1648. The standard survey of acquisition strategies — the
entropy sampling in §4 is the simplest member of a large family.

### 📄 Research connection — Interactive bioimage annotation
Berg, S. et al. (2019). *ilastik: interactive machine learning for (bio)image analysis.* Nature
Methods, 16, 1226–1232. The production version of the human-in-the-loop idea in §5, built around
classical pixel classifiers specifically so that retraining after a correction is instant. napari
(napari.org) plays the same role as an extensible viewer for both classical and deep-learning
annotation workflows.

At production scale, the same ideas apply directly to a 3-D or whole-slide **segmentation**
network (Module 19) rather than a per-image classifier: an uncertainty *map* highlights the exact
pixels or regions an annotator should check first, instead of a single number per image.

## 📌 Key takeaways

- A prediction without an uncertainty estimate is an incomplete answer — "what" without "how sure".
- Aleatoric uncertainty (noisy data) does not shrink with more training data; epistemic
  uncertainty (the model's inexperience) does.
- Calibration — whether stated confidence matches empirical accuracy — is measurable with a
  reliability diagram and the expected calibration error, and is not the same thing as accuracy.
- MC-dropout and deep ensembles are two practical, checkable ways to turn one deterministic-
  looking network into an uncertainty estimate.
- Active learning uses that uncertainty estimate to decide which unlabeled images to annotate
  next, reaching a target accuracy with measurably fewer labels than random annotation.
- Tools like napari and ilastik turn this into a real workflow by putting a human in the loop in
  place of an oracle.

## ✅ Checkpoint

**1. A model reports 95% confidence on every image in a new, out-of-distribution batch, and gets
about 60% of them right. Is this an aleatoric or an epistemic uncertainty problem, and how would
you tell?**

<details><summary>Answer</summary>

It looks epistemic: the model has not seen data like this batch before, so it is extrapolating
confidently into territory it knows nothing about — exactly what Module 34 calls dataset shift.
You would check by seeing whether an ensemble trained on the same data disagrees a lot on this
batch (epistemic) versus agrees but is just wrong (more consistent with the batch being
genuinely ambiguous, i.e. aleatoric). The fix also differs: epistemic uncertainty calls for
retraining or fine-tuning on examples from the new distribution; aleatoric uncertainty would not
be fixed by more labels of the same kind.
</details>

**2. Why does MC-dropout require leaving the network in "training mode" at test time, and what
would go wrong if the network also used BatchNorm layers?**

<details><summary>Answer</summary>

"Training mode" is what keeps Dropout layers stochastic (randomly zeroing activations each pass);
in "eval mode" Dropout becomes a fixed, deterministic rescaling and every pass gives the identical
answer, so there is no spread to measure. BatchNorm layers, however, also behave differently in
training mode — they use the *current batch's* statistics rather than the fixed running
statistics learned during training — which injects an unwanted extra source of randomness (or
outright breaks on a batch size of 1). In practice you either avoid BatchNorm in MC-dropout
models, replace it with a batch-independent alternative, or set those specific layers back to
eval mode by hand while leaving Dropout layers in training mode.
</details>

**3. In the active-learning experiment, what would you expect to happen to the gap between the
two curves if `N_SEED_LABELS` started much larger (say, 200 per class instead of 2)?**

<details><summary>Answer</summary>

The gap should shrink. Once the model has already seen plenty of examples of every class, its
epistemic uncertainty is low everywhere in the pool, so there is little informative signal left
for uncertainty sampling to exploit over random selection — active learning helps most in the
label-scarce regime, which is exactly when real annotation budgets are tightest.
</details>

## 🧠 Questions & discussion

- A model is 99% confident and wrong. Who should be held responsible: the model, the person who
  deployed it without checking calibration, or the person who used its output without checking
  the confidence score? Does your answer change if the confidence score was never surfaced to the
  end user in the first place?
- Active learning optimizes for the model's accuracy. Could optimizing purely for "what helps the
  model most" ever conflict with what a biologist actually needs labeled (e.g., rare but
  biologically critical phenotypes the model is *not* uncertain about because it has never even
  encountered one)? How would you change the acquisition function to account for that?
- Ensembles and MC-dropout both cost more compute than a single deterministic forward pass. In a
  live microscope pipeline processing frames in real time, when is that cost worth paying and when
  is it not?

## 🚀 Bonus — going further

- Implement a BALD-style acquisition function (disagreement *among* individual MC-dropout passes,
  not just entropy of their average) and compare its learning curve to plain entropy sampling.
- Repeat the calibration experiment (§1) after artificially training on a class-imbalanced dataset
  (many "round" cells, few "ring" cells). Does the reliability diagram get worse specifically for
  the rare class?
- Simulate a more adversarial active-learning setting: a pool that is 90% one easy class and 10%
  a hard, ambiguous class. Does entropy sampling over-focus on the easy class's occasional noisy
  example instead of the systematically harder one? This is a well-known failure mode of naive
  uncertainty sampling called "querying by noise".

## 📝 Lab note

Record, in your own words:

1. One number: the expected calibration error you measured for the base model, and whether you
   would call it "well calibrated" for a real lab decision.
2. One sentence distinguishing, in your own example (not one from this notebook), aleatoric from
   epistemic uncertainty.
3. One claim backed by a number: at what label count did uncertainty sampling first beat random
   sampling's held-out accuracy, and by how much?

## 📚 References

1. Gal, Y. & Ghahramani, Z. (2016). Dropout as a Bayesian Approximation: Representing Model
   Uncertainty in Deep Learning. *ICML*.
2. Lakshminarayanan, B., Pritzel, A. & Blundell, C. (2017). Simple and Scalable Predictive
   Uncertainty Estimation using Deep Ensembles. *NeurIPS*.
3. Guo, C., Pleiss, G., Sun, Y. & Weinberger, K. Q. (2017). On Calibration of Modern Neural
   Networks. *ICML*.
4. Settles, B. (2009). Active Learning Literature Survey. University of Wisconsin-Madison,
   Computer Sciences Technical Report 1648.
5. Houlsby, N. et al. (2011). Bayesian Active Learning for Classification and Preference Learning.
   *arXiv:1112.5745* (the BALD acquisition function mentioned in the Bonus section).
6. Berg, S. et al. (2019). ilastik: interactive machine learning for (bio)image analysis. *Nature
   Methods*, 16, 1226–1232.
7. Ouyang, W. & Zimmer, C. (2017). Deep learning in microscopy image analysis: a survey (review
   covering uncertainty-aware approaches in bioimage segmentation). *IEEE TVCG*.
8. napari contributors. napari: a multi-dimensional image viewer for Python. https://napari.org

## ➡️ Next

Module 33 tackles a completely different kind of scale problem: what happens when your image is
not 160 × 160 pixels but a multi-gigabyte 3-D or 4-D volume that does not fit in memory at all.